In [67]:
import pandas as pd
import yfinance as yf
import numpy as np

### Data

In [68]:
df = pd.read_csv("nq-15min.csv")

df.set_index("Datetime", inplace=True)

df = df[["Open", "High", "Low", "Close"]]

df.index = (
    pd.to_datetime(df.index, unit='ms', utc=True).tz_convert('America/New_York')
)

In [69]:
# Ticker = "NQ=F"

# df = yf.download(Ticker, period="max", interval="15m")

# df.columns.names = [None, None]

# df.columns = df.columns.get_level_values(0)

# df = df.drop(columns=["Volume"])

In [70]:
df

,Open,High,Low,Close
Datetime,,,,
2019-12-31 19:00:00-05:00,8750.160,8750.160,8750.160,8750.160
2019-12-31 19:15:00-05:00,8750.160,8750.160,8750.160,8750.160
2019-12-31 19:30:00-05:00,8750.160,8750.160,8750.160,8750.160
2019-12-31 19:45:00-05:00,8750.160,8750.160,8750.160,8750.160
2019-12-31 20:00:00-05:00,8750.160,8750.160,8750.160,8750.160
...,...,...,...,...
2026-08-04 17:45:00-04:00,29787.043,29787.043,29787.043,29787.043
2026-08-04 18:00:00-04:00,29674.810,29711.131,29652.487,29709.965
2026-08-04 18:15:00-04:00,29709.632,29718.589,29685.543,29707.531


### Vars

In [71]:
RANGE_LENGTH = 1
RRR = 2
TARGET_CANDLE_TIME = "09:15"
SL_RANGE_PCT = 2

# Range length: 1, RRR: 2.0, Candle time: 09:15, SL range percentage: 2

### Strategy

In [72]:
class RangeBreakout:
    def __init__(
            self,
            sl_range_pct=SL_RANGE_PCT,
            range_length=RANGE_LENGTH,
            rrr=RRR,
            target_candle_time=TARGET_CANDLE_TIME,
            balance=1000,
            risk_per_trade=10,

    ):
        self.sl_range_pct = sl_range_pct
        self.range_length = range_length
        self.rrr = rrr
        self.target_candle_time = target_candle_time
        self.balance = balance
        self.risk_per_trade = risk_per_trade

    def run(self, df):
        high = df["High"].to_numpy()
        low = df["Low"].to_numpy()

        PnL = np.full(len(df), 0.0)

        cum_pnl = 0

        sl_range_pct = self.sl_range_pct

        for root in range(self.range_length - 1, len(df)):

            if self.balance + cum_pnl * self.risk_per_trade < 0: break

            if df.index[root].strftime("%H:%M") != self.target_candle_time: continue

            range_high = high[root - self.range_length + 1:root + 1].max()
            range_low = low[root - self.range_length + 1:root + 1].min()

            range_width = range_high - range_low

            is_long = False
            is_short = False

            for i in range(root + 1, len(df)):
                if not is_long and not is_short:

                    if high[i] >= range_high and low[i] > range_high - sl_range_pct * range_width : is_long = True

                    elif low[i] <= range_low and high[i] < range_low +  sl_range_pct * range_width : is_short = True

                    elif low[i] > range_low and high[i] < range_high: continue # trade is not triggered yet

                    else: # Trade triggered and SL hit right after it
                        PnL[root] = -1
                        cum_pnl += -1
                        break  
                if is_long:
                    if low[i] <= range_high - sl_range_pct * range_width: # SL hit
                        PnL[root] = -1
                        cum_pnl += -1
                        break  

                    if high[i] - range_high > self.rrr * range_width * sl_range_pct: # TP hit
                        PnL[root] = self.rrr
                        cum_pnl += self.rrr
                        break

                elif is_short:
                    if high[i] >= range_low +  sl_range_pct * range_width: # SL hit
                        PnL[root] = -1
                        cum_pnl += -1
                        break

                    if range_low - low[i] > self.rrr * range_width * sl_range_pct: # TP hit
                        PnL[root] = self.rrr
                        cum_pnl += self.rrr
                        break
        return pd.DataFrame(
            {
                "PnL": PnL,
                "Cumulative_PnL": np.cumsum(PnL),
                "Balance": self.balance + np.cumsum(PnL) * self.risk_per_trade

             },
            index=df.index
        )

In [73]:
backtest = RangeBreakout()

result = backtest.run(df)

result

,PnL,Cumulative_PnL,Balance
Datetime,,,
2019-12-31 19:00:00-05:00,0.0,0.0,1000.0
2019-12-31 19:15:00-05:00,0.0,0.0,1000.0
2019-12-31 19:30:00-05:00,0.0,0.0,1000.0
2019-12-31 19:45:00-05:00,0.0,0.0,1000.0
2019-12-31 20:00:00-05:00,0.0,0.0,1000.0
...,...,...,...
2026-08-04 17:45:00-04:00,0.0,402.0,5020.0
2026-08-04 18:00:00-04:00,0.0,402.0,5020.0
2026-08-04 18:15:00-04:00,0.0,402.0,5020.0


### Result plot

In [74]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=result.index,
    y=result["Balance"],
    mode="lines",
    name="Balance"
))

fig.update_layout(
    title="Balance Over Time",
    xaxis_title="Time",
    yaxis_title="Account Balance Over Time ($)",
    hovermode="x unified",
    template="plotly_white"
)

fig.show()

### Stats

In [75]:
def backtest_stats(result_df):
    result_df = result_df.copy()

    # --------------------------------------------------
    # 1. Return %
    # --------------------------------------------------
    initial_balance = result_df["Balance"].iloc[0]

    final_balance = result_df["Balance"].iloc[-1]
    net_profit = final_balance - initial_balance
    return_pct = net_profit / initial_balance * 100

    # --------------------------------------------------
    # 3. Balance Drawdown
    # --------------------------------------------------
    balance_peak = result_df["Balance"].cummax()

    balance_dd = balance_peak - result_df["Balance"]
    balance_dd_pct = balance_dd / balance_peak * 100

    max_balance_dd = balance_dd.max()
    max_balance_dd_pct = balance_dd_pct.max()


    # --------------------------------------------------
    # 3. Win rate
    # --------------------------------------------------
    _winning_trades = result_df["PnL"][result_df["PnL"] > 0].count()
    _losing_trades = result_df["PnL"][result_df["PnL"] < 0].count()

    # --------------------------------------------------
    # 3. Win rate
    # --------------------------------------------------
    s = result_df["PnL"][result_df["PnL"] != 0]

    wins = s > 0
    losses = s < 0

    win_streaks = wins.groupby((wins != wins.shift()).cumsum()).sum()
    loss_streaks = losses.groupby((losses != losses.shift()).cumsum()).sum()

    max_win_streak = int(win_streaks.max())
    max_loss_streak = int(loss_streaks.max())

    # --------------------------------------------------
    # Return results
    # --------------------------------------------------
    return {
        "Initial Balance": initial_balance,
        "Final Balance": final_balance,
        "Net Profit": net_profit,
        "Return %": return_pct,

        "Max Balance DD": max_balance_dd,
        "Max Balance DD %": max_balance_dd_pct,

        "Win rate": 100 * _winning_trades/(_winning_trades + _losing_trades),

        "Winning trades": _winning_trades,
        "Losing trades": _losing_trades,

        "Max win streak": max_win_streak,
        "Max loss streak": max_loss_streak
    }

def print_stats(stats):
    for name, value in stats.items():
        print(f"{name:20s}: {value:.4f}")


In [76]:
print_stats(backtest_stats(result))

Initial Balance     : 1000.0000
Final Balance       : 5020.0000
Net Profit          : 4020.0000
Return %            : 402.0000
Max Balance DD      : 190.0000
Max Balance DD %    : 10.2804
Win rate            : 39.8923
Winning trades      : 815.0000
Losing trades       : 1228.0000
Max win streak      : 5.0000
Max loss streak     : 15.0000


### 7. (Optional) Parameter sweep

In [77]:
# from itertools import product

# # TARGET_CANDLE_TIME = "09:15"

# range_lengths = [i for i in range(1, 6)]
# rrrs = [i * 0.5 for i in range(1, 7)]
# times = ["09:00", "09:15", "09:30", "09:45", "10:00", "10:15", "10:30", "10:45"]
# sl_range_pcts = [0.5, 1, 1.5, 2]

# results = {}

# for rl, rr, ct, srp in product(range_lengths, rrrs, times, sl_range_pcts):
#     _backtest = RangeBreakout(range_length=rl, rrr=rr, target_candle_time=ct, sl_range_pct=srp)
#     results[(rl, rr, ct, srp)] = backtest_stats(_backtest.run(df))

# best = sorted(
#     results.items(),
#     key=lambda x: x[1]["Max Balance DD %"]
# )[:3]

# # only three best ones
# for (rl, rr, ct, srp), backtest_result in best:
#     print(f"Range length: {rl}, RRR: {rr}, Candle time: {ct}, SL range percentage: {srp}")
#     print_stats(backtest_result)
#     print("\n\n--------------------------------------------------\n\n")

# print("Done!")